# Flowcast: train and benchmark on an AMD GPU

Trains the Flowcast ensemble (four flow networks and four particle networks) on a GPU with ROCm, measures it against simulated rooms it never saw, times it, and packages the trained models.

Run the cells in order. The only thing to set is `DATA_URL` in the third cell.

## 1. What is this machine?

In [ ]:
!rocm-smi 2>/dev/null || nvidia-smi 2>/dev/null || echo "no GPU tool found"
import torch, os
print("torch", torch.__version__, "| ROCm", getattr(torch.version, "hip", None), "| GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(p.name, f"{p.total_memory / 2**30:.0f} GB")
print("processor threads:", os.cpu_count())
!df -h . | tail -n 1

## 2. Get the code and the simulated rooms

In [ ]:
DATA_URL = "https://github.com/MrtnOmwenga/flowcast/releases/download/data-v3/flowcast-cleanroom-panels-v3.tar"

!test -d flowcast || git clone --depth 1 https://github.com/MrtnOmwenga/flowcast.git
%cd flowcast
!pip install -q numpy
!mkdir -p data && test -d data/cases-v3 || (curl -L --fail -o data/rooms.tar "$DATA_URL" && tar -xf data/rooms.tar -C data && rm data/rooms.tar)
!ls data/cases-v3 | wc -l

## 3. Train the ensemble

All eight networks train at the same time on the one GPU. Each seed gives an independently trained copy; their disagreement is the confidence reading.

In [ ]:
import subprocess, time, pathlib
pathlib.Path("data/logs").mkdir(parents=True, exist_ok=True)
pathlib.Path("data/models").mkdir(parents=True, exist_ok=True)
pathlib.Path("data/models/meta.json").write_text('{"people": true}')
started = time.time()
jobs = []
for seed in range(4):
    for task, epochs in (("flow", 80), ("particles", 40)):
        log = open(f"data/logs/{task}-{seed}.log", "w")
        jobs.append(subprocess.Popen(
            ["python", "-m", "flowcast.model.train", "--task", task, "--seed", str(seed), "--data", "data/cases-v3",
             "--out", f"data/models/{task}/{seed}", "--epochs", str(epochs), "--workers", "2"],
            stdout=log, stderr=subprocess.STDOUT))
while any(j.poll() is None for j in jobs):
    time.sleep(60)
    done = sum(j.poll() is not None for j in jobs)
    last = subprocess.run("tail -q -n 1 data/logs/flow-0.log data/logs/particles-0.log", shell=True, capture_output=True, text=True).stdout
    print(f"{(time.time() - started) / 60:5.1f} min, {done}/8 finished\n{last}", flush=True)
print(f"all eight networks trained in {(time.time() - started) / 60:.1f} minutes; exit codes {[j.returncode for j in jobs]}")

## 4. How accurate is it on rooms it never saw?

In [ ]:
!python -m flowcast.model.evaluate --data data/cases-v3 --models data/models --out data/accuracy.json

## 5. How fast is it on this GPU?

In [ ]:
!python -m flowcast.model.benchmark --models data/models --out data/benchmark.json

## 6. Package the results for download

In [ ]:
!tar -czf flowcast-models.tar.gz data/models data/accuracy.json data/benchmark.json data/logs
!ls -lh flowcast-models.tar.gz